In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_0.pickle

In [ ]:
%%PandasProfile
### cell 23 ###

def grab_subset_of_data_35(df, question_of_interest):
    # Vectorized column filtering and renaming without an explicit full copy
    mask = df.columns.str.contains(question_of_interest)
    cols = df.columns[mask]
    # Derive new column names by splitting at the last '-'
    new_names = [c.rsplit('-', 1)[-1].lstrip() for c in cols]
    df_sub = df.loc[:, cols]
    df_sub.columns = new_names
    return df_sub.dropna(how='all', subset=new_names)


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(question_of_interest, include_2017=None):
    # Define (dataframe, year) pairs in chronological order
    sources = [
        (responses_df_2018_cell10, '2018'),
        (responses_df_2019_cell10, '2019'),
        (responses_df_2020,        '2020'),
        (responses_df_2021,        '2021'),
        (responses_df_2022_cell10, '2022'),
    ]
    if include_2017 is not None:
        sources.insert(0, (responses_df_2017, '2017'))
    # Subset, rename, dropna, and assign year in one comprehension
    dfs = [
        grab_subset_of_data_35(src_df, question_of_interest).assign(year=yr)
        for src_df, yr in sources
    ]
    # Concatenate once with a fresh integer index
    df_combined = pd.concat(dfs, ignore_index=True)
    # Count non-null responses per year
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Cast counts to int and set 'year' as index for alignment
    df_counts_int = df_counts.astype(int).set_index('year')
    # Compute total respondents per year once
    totals = df['year'].value_counts().reindex(df_counts_int.index)
    # Divide and multiply by 100, then bring 'year' back as a column
    percentages = df_counts_int.div(totals, axis=0).mul(100).reset_index()
    return percentages

# ---- Pipeline ----
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'

language_df_combined, language_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )

language_df_combined_percentages = \
    convert_df_of_counts_to_percentages_35(
        language_df_combined,
        language_df_combined_counts
    )

# Select the languages of interest and reshape
languages = ['Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']

df_cell35 = (
    language_df_combined_percentages
        .loc[:, ['year'] + languages]
        .melt(id_vars=['year'], value_vars=languages)
        .sort_values(by=['year', 'value'])
        .rename(columns={'variable': ''})
)

# Display info
df_cell35.info()